In [0]:
import json
import os
import re
import hashlib
import pandas as pd

CATALOG = "workspace"
SCHEMA = "default"
TABLE = "legal_rag_corpus"
VOLUME = "bharat_bricks_hacks"
VOL_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Path to the IndicLegalQA JSON file — try multiple locations
DATASET_PATH_VOLUME = f"{VOL_PATH}/datasets/IndicLegalQA Dataset_10K_Revised.json"
DATASET_PATH_VOLUME_ALT = f"{VOL_PATH}/IndicLegalQA Dataset_10K_Revised.json"
DATASET_PATH_REPO = "/Workspace/Users/saisandeshk@iisc.ac.in/bharat-bricks-hacks/datasets/IndicLegalQA Dataset_10K_Revised.json"
DATASET_PATH_PARENT = "/Workspace/Users/saisandeshk@iisc.ac.in/DataBricks/datasets/IndicLegalQA Dataset_10K_Revised.json"

print(f"✅ Config: {CATALOG}.{SCHEMA}.{TABLE}")
print(f"   Volume: {VOL_PATH}")

In [0]:
# Try multiple paths
dataset_file = None
for path in [DATASET_PATH_VOLUME, DATASET_PATH_VOLUME_ALT, DATASET_PATH_REPO, DATASET_PATH_PARENT]:
    if os.path.exists(path):
        dataset_file = path
        break

if dataset_file is None:
    # Try dbutils to copy from workspace
    try:
        dbutils.fs.cp(
            f"file:{DATASET_PATH_REPO}",
            f"file:/tmp/indiclegal_qa.json"
        )
        dataset_file = "/tmp/indiclegal_qa.json"
    except Exception:
        raise FileNotFoundError(
            f"IndicLegalQA dataset not found at:\n"
            f"  {DATASET_PATH}\n"
            f"  {DATASET_PATH_REPO}\n"
            f"Upload the JSON file to the Volume or Workspace."
        )

with open(dataset_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"✅ Loaded {len(raw_data)} QA pairs from {dataset_file}")

# Quick stats
case_names = set(e["case_name"] for e in raw_data)
print(f"   Unique cases: {len(case_names)}")

In [0]:
from collections import defaultdict

# Group QA pairs by case
cases = defaultdict(list)
for entry in raw_data:
    cases[entry["case_name"]].append(entry)

print(f"Grouped into {len(cases)} cases")

# --- Strategy A: Per-case summary chunks ---
summary_chunks = []
for case_name, qa_pairs in cases.items():
    date = qa_pairs[0].get("judgement_date", "")

    # Build a rich summary text
    qa_text_parts = []
    for i, qa in enumerate(qa_pairs, 1):
        qa_text_parts.append(f"Q{i}: {qa['question']}\nA{i}: {qa['answer']}")

    full_text = (
        f"Supreme Court of India — {case_name}"
        f"{f' ({date})' if date else ''}\n\n"
        f"Key legal points from this judgment:\n\n"
        + "\n\n".join(qa_text_parts)
    )

    # Generate deterministic chunk_id from case name
    case_hash = hashlib.md5(case_name.encode()).hexdigest()[:8]
    chunk_id = f"ILQA_CASE_{case_hash}"

    summary_chunks.append({
        "chunk_id": chunk_id,
        "source": f"IndicLegalQA_SC_{case_name[:80]}",
        "doc_type": "sc_judgment_qa",
        "title": f"SC Judgment: {case_name} ({date})" if date else f"SC Judgment: {case_name}",
        "text": full_text[:4000],  # cap at 4000 chars for embedding quality
    })

print(f"✅ Strategy A: {len(summary_chunks)} case summary chunks")

# --- Strategy B: Individual QA chunks ---
qa_chunks = []
for entry in raw_data:
    case_name = entry["case_name"]
    date = entry.get("judgement_date", "")
    q = entry["question"]
    a = entry["answer"]

    text = (
        f"Supreme Court of India — {case_name}"
        f"{f' ({date})' if date else ''}\n"
        f"Question: {q}\n"
        f"Answer: {a}"
    )

    # Deterministic chunk_id
    qa_hash = hashlib.md5(f"{case_name}:{q}".encode()).hexdigest()[:10]
    chunk_id = f"ILQA_QA_{qa_hash}"

    qa_chunks.append({
        "chunk_id": chunk_id,
        "source": f"IndicLegalQA_SC_{case_name[:80]}",
        "doc_type": "sc_judgment_qa",
        "title": f"SC: {case_name[:60]} — {q[:80]}",
        "text": text,
    })

print(f"✅ Strategy B: {len(qa_chunks)} individual QA chunks")

In [0]:
all_new_chunks = summary_chunks + qa_chunks
df = pd.DataFrame(all_new_chunks)

print(f"Total new chunks to ingest: {len(df)}")
print(f"  Case summaries: {len(summary_chunks)}")
print(f"  Individual QAs:  {len(qa_chunks)}")
print(f"\nDoc type distribution:")
print(df["doc_type"].value_counts().to_string())
print(f"\nSample chunk (summary):")
print(df.iloc[0]["text"][:500])

In [0]:
# Write to Delta — APPEND to existing corpus
sdf = spark.createDataFrame(df)
sdf.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")

print(f"✅ Appended {len(df)} chunks to {CATALOG}.{SCHEMA}.{TABLE}")

In [0]:
total_df = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}")
print(f"Total corpus size: {total_df.count()} chunks")
print(f"\nDoc type distribution:")
total_df.groupBy("doc_type").count().orderBy("count", ascending=False).show(20, truncate=False)

print(f"\nSample SC judgment QA chunks:")
total_df.filter("doc_type = 'sc_judgment_qa'").select("chunk_id", "title").show(10, truncate=False)
